In [ ]:
import sys
from pathlib import Path

# If running inside the repo:
sys.path.append(".")

import pandas as pd
import numpy as np

from src.data.preprocess import preprocess_align
from src.data.splits import prepare_splits_and_scalers

In [ ]:
A = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Alexander_upto_17.csv")
J = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Jones_upto_15_MIRRORS.csv")
H = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/HomflyPt_upto_15_MIRRORS.csv")

In [ ]:
aligned = preprocess_align(
    A,
    J,
    H,
    max_cross=15,
    drop_mirrors=True,
    drop_signature_mismatches=True,
    verbose=True,
)

prepared = prepare_splits_and_scalers(
    aligned["X_A"],
    aligned["X_J"],
    aligned["X_H"],
    aligned["metadata"],
    seed=42,
)

In [ ]:
A_tr = prepared["A"]["train"]
A_va = prepared["A"]["val"]
A_te = prepared["A"]["test"]

J_tr = prepared["J"]["train"]
J_va = prepared["J"]["val"]
J_te = prepared["J"]["test"]

H_tr = prepared["H"]["train"]
H_va = prepared["H"]["val"]
H_te = prepared["H"]["test"]

metadata_train = prepared["metadata"]["train"]
metadata_val = prepared["metadata"]["val"]
metadata_test = prepared["metadata"]["test"]

train_idx = prepared["indices"]["train"]
val_idx = prepared["indices"]["val"]
test_idx = prepared["indices"]["test"]

In [ ]:
print("Aligned N before signature class filter:", aligned["debug"]["aligned_rows_final"])
print("Final N after signature class filter:", len(prepared["metadata"]["all"]))
print("Dropped signature classes:", prepared["dropped_signature_classes"])

print("A:", A_tr.shape, A_va.shape, A_te.shape)
print("J:", J_tr.shape, J_va.shape, J_te.shape)
print("H:", H_tr.shape, H_va.shape, H_te.shape)

print("Test signature counts:")
print(metadata_test["signature"].value_counts().sort_index())

## Raw coeff - bulk_signature_decoding

In [ ]:
from src.evaluation.bulk_decoding import eval_multinomial_lr_split, majority_baseline
from src.features.summary_features import all_summary_features

y_train = metadata_train["signature"].to_numpy()
y_test = metadata_test["signature"].to_numpy()

raw_results = {
    "Jones": eval_multinomial_lr_split(J_tr, y_train, J_te, y_test),
    "Alexander": eval_multinomial_lr_split(A_tr, y_train, A_te, y_test),
    "HOMFLY": eval_multinomial_lr_split(H_tr, y_train, H_te, y_test),
}

for k, res in raw_results.items():
    print(f"{k:9s} Acc={res['accuracy']:.3f} Macro-F1={res['macro_f1']:.3f}")

print("Majority baseline:", majority_baseline(y_train, y_test))

# Confounder/summary features example for HOMFLY
Xtr_conf = all_summary_features(H_tr)
Xte_conf = all_summary_features(H_te)

conf_res = eval_multinomial_lr_split(Xtr_conf, y_train, Xte_conf, y_test)
print("HOMFLY summary/confounder-only:", conf_res["accuracy"], conf_res["macro_f1"])

## PCA + Autoencoder + NRE + tail tables + AE/PCA latent probes.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

from src.models.autoencoder import train_autoencoder, encode_with_model, set_global_seed
from src.evaluation.reconstruction_scores import nre_from_pca, nre_from_model
from src.evaluation.tail_metrics import compute_tail_table
from src.evaluation.bulk_decoding import eval_multinomial_lr_split

SEED = 42
set_global_seed(SEED)

y_train = metadata_train["signature"].to_numpy().astype(int)
y_val = metadata_val["signature"].to_numpy().astype(int)
y_test = metadata_test["signature"].to_numpy().astype(int)

X_store = {
    "Alexander": {"train": A_tr, "val": A_va, "test": A_te},
    "Jones": {"train": J_tr, "val": J_va, "test": J_te},
    "HOMFLY": {"train": H_tr, "val": H_va, "test": H_te},
}

ARCH = {
    "Alexander": {"latent": 16, "widths": [64, 32, 32]},
    "Jones": {"latent": 16, "widths": [128, 64, 64]},
    "HOMFLY": {"latent": 16, "widths": [256, 128, 128]},
}

D_PCA = 16
TAUS_MAIN = [0.95, 0.99]
S_MAIN = [8, 10]

ae_models = {}
encoders = {}
pca_models = {}

scores_all = {}
latent_probe_rows = []

for inv in ["Alexander", "Jones", "HOMFLY"]:
    print(f"\n=== {inv} ===")

    Xtr = np.asarray(X_store[inv]["train"], dtype=np.float32)
    Xva = np.asarray(X_store[inv]["val"], dtype=np.float32)
    Xte = np.asarray(X_store[inv]["test"], dtype=np.float32)

    scores_all[inv] = {"PCA": {}, "AE": {}}

    # -------------------------
    # PCA fit on train only
    # -------------------------
    print("Fitting PCA...")
    pca = PCA(n_components=D_PCA, random_state=SEED)
    Z_pca_tr = pca.fit_transform(Xtr).astype(np.float32)
    Z_pca_te = pca.transform(Xte).astype(np.float32)

    pca_models[inv] = pca

    scores_all[inv]["PCA"]["test"] = nre_from_pca(pca, Xte)

    pca_probe = eval_multinomial_lr_split(
        Z_pca_tr,
        y_train,
        Z_pca_te,
        y_test,
        seed=SEED,
    )

    latent_probe_rows.append({
        "Invariant": inv,
        "Representation": f"PCA_d{D_PCA}",
        "Accuracy": pca_probe["accuracy"],
        "MacroF1": pca_probe["macro_f1"],
        "n_iter": pca_probe["n_iter"],
    })

    print(
        f"PCA probe: Acc={pca_probe['accuracy']:.3f}, "
        f"Macro-F1={pca_probe['macro_f1']:.3f}"
    )

    # -------------------------
    # Autoencoder train on train, early stopping on val
    # -------------------------
    print("Training autoencoder...")
    ae, enc, history = train_autoencoder(
        Xtr,
        Xva,
        latent_size=ARCH[inv]["latent"],
        widths=ARCH[inv]["widths"],
        learning_rate=5e-5,
        batch_size=128,
        epochs=30,
        patience=5,
        seed=SEED,
        verbose=0,
    )

    ae_models[inv] = ae
    encoders[inv] = enc

    print(
        f"AE epochs ran: {len(history.history['loss'])}, "
        f"best val loss: {np.min(history.history['val_loss']):.6g}"
    )

    # AE NRE on test
    scores_all[inv]["AE"]["test"] = nre_from_model(
        ae,
        Xte,
        batch_size=2048,
    )

    # AE latent probe
    Z_ae_tr = encode_with_model(enc, Xtr, batch_size=2048).astype(np.float32)
    Z_ae_te = encode_with_model(enc, Xte, batch_size=2048).astype(np.float32)

    ae_probe = eval_multinomial_lr_split(
        Z_ae_tr,
        y_train,
        Z_ae_te,
        y_test,
        seed=SEED,
    )

    latent_probe_rows.append({
        "Invariant": inv,
        "Representation": f"AE_d{ARCH[inv]['latent']}",
        "Accuracy": ae_probe["accuracy"],
        "MacroF1": ae_probe["macro_f1"],
        "n_iter": ae_probe["n_iter"],
    })

    print(
        f"AE latent probe: Acc={ae_probe['accuracy']:.3f}, "
        f"Macro-F1={ae_probe['macro_f1']:.3f}"
    )

latent_probe_df = pd.DataFrame(latent_probe_rows)
display(latent_probe_df)

### Tail enrichment PCA NRE and AE NRE

In [ ]:
rows = []

for inv in ["Alexander", "Jones", "HOMFLY"]:
    for method in ["PCA", "AE"]:
        sc_test = scores_all[inv][method]["test"]

        df_tail = compute_tail_table(
            scores=sc_test,
            y_signature=y_test,
            s_list=S_MAIN,
            taus=TAUS_MAIN,
        )

        df_tail["Invariant"] = inv
        df_tail["Score"] = f"NRE_{method}"
        df_tail["Split"] = "test"
        df_tail["Scope"] = "MAIN"

        rows.append(df_tail)

df_master = pd.concat(rows, ignore_index=True)

df_master = df_master[
    [
        "Invariant",
        "Score",
        "Split",
        "Scope",
        "Target",
        "Tau",
        "AUROC",
        "AUPRC",
        "Enrichment",
        "Positives",
        "Tail_captured",
        "Tail_size",
        "N",
        "Base_rate",
    ]
].sort_values(["Invariant", "Score", "Target", "Tau"])

display(df_master)

### Y-12

In [ ]:
rows_y12 = []

for method in ["PCA", "AE"]:
    sc_test = scores_all["Jones"][method]["test"]

    df_y12 = compute_tail_table(
        scores=sc_test,
        y_signature=y_test,
        s_list=[12],
        taus=[0.99],
    )

    df_y12["Invariant"] = "Jones"
    df_y12["Score"] = f"NRE_{method}"
    df_y12["Split"] = "test"
    df_y12["Scope"] = "Y12_STRESS"

    rows_y12.append(df_y12)

df_jones_y12 = pd.concat(rows_y12, ignore_index=True)
display(df_jones_y12)

df_jones_y12.to_csv("jones_Y12_appendix.csv", index=False)

## Ablation

In [ ]:
from src.evaluation.ablation import run_jones_ae_ablation

y_test = metadata_test["signature"].to_numpy().astype(int)

df_ablation = run_jones_ae_ablation(
    X_train=J_tr,
    X_val=J_va,
    X_test=J_te,
    y_test=y_test,
    widths=[128, 64, 64],
    latent_dims=[8, 16, 32],
    seeds=[42, 123, 999],
    score_types=["NRE", "SSE"],
    tau=0.99,
    target_s=10,
    learning_rate=5e-5,
    batch_size=128,
    epochs=30,
    patience=5,
    pred_batch_size=2048,
)

df_ablation.to_csv("jones_ae_ablation_stability.csv", index=False)
display(df_ablation)

## CDF

In [ ]:
from src.evaluation.distribution_diagnostics import (
    run_distribution_diagnostics,
    plot_ccdf_row,
)

from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT_TABLES = Path("results/tables")
OUT_FIGS = Path("results/figures")

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGS.mkdir(parents=True, exist_ok=True)

score_dict = {
    "Alexander": scores_all["Alexander"]["AE"]["test"],
    "Jones": scores_all["Jones"]["AE"]["test"],
    "HOMFLY": scores_all["HOMFLY"]["AE"]["test"],
}

df_pw, fits = run_distribution_diagnostics(
    score_dict=score_dict,
    sample_max=None,   # usa todo; para probar rápido puedes poner 100000
    n_boot=500,        # para probar rápido puedes poner 100
    seed=42,
)

df_pw.to_csv(OUT_TABLES / "distribution_diagnostics_AE_test_seed42.csv", index=False)
display(df_pw)

plot_ccdf_row(
    fits=fits,
    out_pdf=OUT_FIGS / "ccdf_all_invariants_AE_test_seed42.pdf",
    out_png=OUT_FIGS / "ccdf_all_invariants_AE_test_seed42.png",
)

display(Image(filename=str(OUT_FIGS / "ccdf_all_invariants_AE_test_seed42.png")))

## counfounders

In [ ]:
import numpy as np
import pandas as pd

from src.evaluation.confounders import (
    support_width_from_coeffs,
    compute_spearman_confounders,
    build_confounder_matrix_scaled_space,
    evaluate_confounder_models_jones,
    enrichment_sensitivity_stable,
    tail_overlap_against_confounders,
)

SEED = 42

y_train = metadata_train["signature"].to_numpy().astype(int)
y_test = metadata_test["signature"].to_numpy().astype(int)

crossing_train = metadata_train["number_of_crossings"].to_numpy()
crossing_test = metadata_test["number_of_crossings"].to_numpy()

width_train = metadata_train["maximum_exponent"].to_numpy() - metadata_train["minimum_exponent"].to_numpy()
width_test = metadata_test["maximum_exponent"].to_numpy() - metadata_test["minimum_exponent"].to_numpy()

nre_A = scores_all["Alexander"]["AE"]["test"]
nre_J = scores_all["Jones"]["AE"]["test"]
nre_H = scores_all["HOMFLY"]["AE"]["test"]

width_H_test = support_width_from_coeffs(H_te, eps=1e-8)

conf_A = compute_spearman_confounders(A_te, nre_A, crossing_test, width_test)
conf_A.insert(0, "Invariant", "Alexander")

conf_J = compute_spearman_confounders(J_te, nre_J, crossing_test, width_test)
conf_J.insert(0, "Invariant", "Jones")

conf_H = compute_spearman_confounders(H_te, nre_H, crossing_test, width_H_test)
conf_H.insert(0, "Invariant", "HOMFLY")

conf_all = pd.concat([conf_A, conf_J, conf_H], ignore_index=True)
conf_all["Domain"] = "test"
conf_all["N_test"] = len(metadata_test)

display(conf_all)
conf_all.to_csv("confounders_spearman_test.csv", index=False)

### confounder-only Jones

In [ ]:
C_train = build_confounder_matrix_scaled_space(
    X_scaled=J_tr,
    width=width_train,
    crossing=crossing_train,
    l0_tol=0.1,
)

C_test = build_confounder_matrix_scaled_space(
    X_scaled=J_te,
    width=width_test,
    crossing=crossing_test,
    l0_tol=0.1,
)

df_conf_models = evaluate_confounder_models_jones(
    C_train=C_train,
    C_test=C_test,
    sig_train=y_train,
    sig_test=y_test,
    targets=(8, 10),
    taus=(0.95, 0.99),
    seed=SEED,
)

display(df_conf_models)
df_conf_models.to_csv("confounder_models_jones_test_consistent.csv", index=False)

### enrichment condicionado por support-width bins

In [ ]:
def make_df_for_conditioned(signature, crossing, support_width, nre, support_width_type):
    return pd.DataFrame(
        {
            "signature": signature,
            "number_of_crossings": crossing,
            "support_width": support_width,
            "support_width_type": support_width_type,
            "NRE_AE": nre,
        }
    )

dfA = make_df_for_conditioned(y_test, crossing_test, width_test, nre_A, "exponent_width")
dfJ = make_df_for_conditioned(y_test, crossing_test, width_test, nre_J, "exponent_width")
dfH = make_df_for_conditioned(y_test, crossing_test, width_H_test, nre_H, "coef_support_proxy")

enrich_A = enrichment_sensitivity_stable(dfA)
enrich_A.insert(0, "Invariant", "Alexander")
enrich_A["support_width_type"] = "exponent_width"

enrich_J = enrichment_sensitivity_stable(dfJ)
enrich_J.insert(0, "Invariant", "Jones")
enrich_J["support_width_type"] = "exponent_width"

enrich_H = enrichment_sensitivity_stable(dfH)
enrich_H.insert(0, "Invariant", "HOMFLY")
enrich_H["support_width_type"] = "coef_support_proxy"

enrich_all = pd.concat([enrich_A, enrich_J, enrich_H], ignore_index=True)
enrich_all["Domain"] = "test"

display(enrich_all.head(30))
enrich_all.to_csv("tail_enrichment_conditioned_test_smoothed.csv", index=False)

### overlap AE vs confounder tail

In [ ]:
df_overlap, df_ae_only = tail_overlap_against_confounders(
    ae_scores=nre_J,
    C_train=C_train,
    C_test=C_test,
    sig_train=y_train,
    sig_test=y_test,
    targets=(8, 10),
    taus=(0.95, 0.99),
    seed=SEED,
)

display(df_overlap)
df_overlap.to_csv("tail_overlap_jones_test.csv", index=False)

if len(df_ae_only) > 0:
    display(df_ae_only.head(20))
    df_ae_only.to_csv("ae_only_tail_positives_jones.csv", index=False)

display(
    df_overlap[
        (df_overlap["Target"] == "Y10") & (np.isclose(df_overlap["Tau"], 0.99))
    ][
        [
            "Jaccard",
            "AE_tail_covered_by_Conf",
            "Conf_tail_covered_by_AE",
            "AE_Enrichment",
            "Conf_Enrichment",
            "AE_pos_in_tail",
            "Conf_pos_in_tail",
            "AE_only_tail_size",
            "AE_only_pos_in_tail",
        ]
    ]
)

### Crossing number

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kruskal
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score

def topk_tail_mask(scores, tau):
    scores = np.asarray(scores, dtype=np.float64).ravel()
    n = len(scores)
    k = int(np.ceil((1.0 - tau) * n))

    order = np.argsort(-scores, kind="mergesort")
    mask = np.zeros(n, dtype=bool)
    mask[order[:k]] = True
    return mask

cross_test = metadata_test["number_of_crossings"].to_numpy().astype(int)

nre_test = {
    "Alexander": np.asarray(scores_all["Alexander"]["AE"]["test"], dtype=np.float64),
    "Jones": np.asarray(scores_all["Jones"]["AE"]["test"], dtype=np.float64),
    "HOMFLY": np.asarray(scores_all["HOMFLY"]["AE"]["test"], dtype=np.float64),
}

TAIL_TAU = 0.99
MIN_GROUP_SIZE = 10

rows_cross = []

for inv, scores in nre_test.items():
    df = pd.DataFrame({
        "cross": cross_test,
        "nre": scores,
    })

    df["is_tail"] = topk_tail_mask(df["nre"].to_numpy(), TAIL_TAU).astype(int)

    rho, rho_p = spearmanr(df["cross"], df["nre"])

    groups = [
        g["nre"].to_numpy()
        for _, g in df.groupby("cross")
        if len(g) >= MIN_GROUP_SIZE
    ]

    if len(groups) >= 2:
        kw_H, kw_p = kruskal(*groups)
    else:
        kw_H, kw_p = np.nan, np.nan

    mi = mutual_info_score(df["cross"], df["is_tail"])
    nmi = normalized_mutual_info_score(df["cross"], df["is_tail"])

    rows_cross.append({
        "Invariant": inv,
        "Domain": "test",
        "N_test": len(df),
        "Tail_tau": TAIL_TAU,
        "Tail_size": int(df["is_tail"].sum()),
        "Spearman_rho_cross_NRE": float(rho),
        "Spearman_p": float(rho_p),
        "Kruskal_H": float(kw_H) if np.isfinite(kw_H) else np.nan,
        "Kruskal_p": float(kw_p) if np.isfinite(kw_p) else np.nan,
        "MI_cross_tail": float(mi),
        "NMI_cross_tail": float(nmi),
        "n_cross_groups": int(df["cross"].nunique()),
    })

df_cross = pd.DataFrame(rows_cross)
display(df_cross)

df_cross.to_csv("crossing_number_nre_tail_test.csv", index=False)
print("Saved: crossing_number_nre_tail_test.csv")

### ODD

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT_FIGS = Path("results/figures")
OUT_FIGS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("progressive_ood_fixed_train_jones.csv")
df = df.sort_values(["test_cross", "horizon", "k_train_max"]).reset_index(drop=True)

def plot_line(ax, sub, xcol, ycol, label):
    x = sub[xcol].to_numpy()
    y = sub[ycol].to_numpy()
    ax.plot(x, y, marker="o", linewidth=1, label=label)

fig = plt.figure(figsize=(7.2, 3.4), dpi=200)
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)

for h in sorted(df["horizon"].unique()):
    sub = df[df["horizon"] == h]
    plot_line(ax1, sub, "test_cross", "probe_acc", label=f"h={h} acc")
    plot_line(ax1, sub, "test_cross", "probe_macroF1", label=f"h={h} macro-F1")

ax1.set_xlabel("Test crossing number")
ax1.set_ylabel("Probe performance (Jones)")
ax1.set_xticks(sorted(df["test_cross"].unique()))
ax1.legend(fontsize=7, frameon=False)

df_auroc = df[np.isfinite(df["pcares_auroc_Y10"].to_numpy())].copy()

for h in sorted(df_auroc["horizon"].unique()):
    sub = df_auroc[df_auroc["horizon"] == h]
    plot_line(ax2, sub, "test_cross", "pcares_auroc_Y10", label=f"h={h} AUROC")

ax2.set_xlabel("Test crossing number")
ax2.set_ylabel(r"PCA-residual AUROC for $Y_{10}$")
ax2.set_xticks(sorted(df["test_cross"].unique()))
ax2.legend(fontsize=7, frameon=False)

plt.tight_layout()

out_png = OUT_FIGS / "fig_progressive_ood_fixedtrain_jones.png"
out_pdf = OUT_FIGS / "fig_progressive_ood_fixedtrain_jones.pdf"

plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()

print("Saved:", out_png)
print("Saved:", out_pdf)